In [ ]:
# Section 1: Environment setup & imports
# Install packages if needed (uncomment to run in a fresh env)
# !pip install -r requirements.txt

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

np.random.seed(42)
RANDOM_STATE = 42

print("Python executable:", sys.executable)

In [ ]:
# Section 2: Load sample dataset
from pathlib import Path
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
train_path = DATA_DIR / 'Training dataset.csv'
test_path = DATA_DIR / 'Test data.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print('Train shape:', train.shape)
print(train.head())
print('\nTest shape:', test.shape)
print(test.head())

In [ ]:
# Section 3: Data cleaning & preprocessing

# Quick checks
assert train.isnull().sum().sum() == 0, 'Unexpected missing values in training data'
assert test.isnull().sum().sum() == 0, 'Unexpected missing values in test data'

# Feature / target split
X = train.drop(columns=['Product Quality'])
y = train['Product Quality']

# Example feature distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, col in zip(axes, X.columns):
    sns.histplot(X[col], ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# Scale example (the notebook shows how to save the scaler)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save a sample scaled dataset
pd.DataFrame(X_scaled, columns=X.columns).head()

In [ ]:
# Section 4: Define core functions and pipeline
from typing import Tuple

def load_data(data_dir: str = DATA_DIR) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(os.path.join(data_dir, 'Training dataset.csv'))
    test = pd.read_csv(os.path.join(data_dir, 'Test data.csv'))
    return train, test


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # placeholder: no missing values in this dataset
    return df.copy()


def transform_features(df: pd.DataFrame, scaler: StandardScaler = None) -> Tuple[pd.DataFrame, StandardScaler]:
    X = df.copy()
    if scaler is None:
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
    else:
        Xs = scaler.transform(X)
    return pd.DataFrame(Xs, columns=X.columns), scaler


def run_pipeline():
    train, test = load_data()
    X = train.drop(columns=['Product Quality'])
    y = train['Product Quality']
    Xs, scaler = transform_features(X)
    # simple train/val
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    Xtr, Xval, ytr, yval = train_test_split(Xs, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    clf.fit(Xtr, ytr)
    preds = clf.predict(Xval)
    from sklearn.metrics import classification_report
    print(classification_report(yval, preds))

# Example call (commented out for fast runs)
# run_pipeline()

In [ ]:
# Section 5: Unit tests for core functions (example)
# We'll write a simple pytest test file in the repo. In VS Code Test Explorer, mark tests folder.

TEST_FILE = os.path.join(ROOT, 'tests_assignment1.py')
with open(TEST_FILE, 'w') as f:
    f.write('''import pandas as pd\nfrom assignment1_pipeline import prepare\n\ndef test_prepare_returns_X_y():\n    df = pd.DataFrame({\n        'Temperature':[1.0,2.0],\n        'Vibration':[3.0,4.0],\n        'Stress index':[5.0,6.0],\n        'Product Quality':[1,2]\n    })\n    X,y = prepare(df)\n    assert list(X.columns) == ['Temperature','Vibration','Stress index']\n    assert len(y) == 2\n''')
print('Wrote simple pytest file:', TEST_FILE)

print('\nTo run tests:')
print('pytest', TEST_FILE)

In [ ]:
# Section 6: Run pipeline and capture outputs
# This cell demonstrates loading the saved best model and running predictions
model_path = os.path.join(ROOT, 'best_model.joblib')
scaler_path = os.path.join(ROOT, 'scaler.joblib')
if os.path.exists(model_path) and os.path.exists(scaler_path):
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    X = train.drop(columns=['Product Quality'])
    Xs = scaler.transform(X)
    from sklearn.model_selection import train_test_split
    Xtr, Xval, ytr, yval = train_test_split(Xs, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    val_preds = model.predict(Xval)
    from sklearn.metrics import confusion_matrix, classification_report
    print('Validation classification report:')
    print(classification_report(yval, val_preds))
    cm = confusion_matrix(yval, val_preds)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel('pred')
    plt.ylabel('true')
    plt.title('Confusion Matrix (validation)')
    plt.show()
else:
    print('Model or scaler not found. Run train_and_tune.py to create them.')

In [ ]:
# Section 7: Visualizations (feature importance)
if os.path.exists(model_path):
    try:
        importances = model.feature_importances_
        fi = pd.Series(importances, index=X.columns).sort_values(ascending=False)
        plt.figure(figsize=(8,4))
        sns.barplot(x=fi.values, y=fi.index)
        plt.title('Feature importances (RandomForest)')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print('Could not plot feature importances:', e)
else:
    print('Model not present to compute feature importances.')

# Additional plots: pairplot sample (commented to avoid heavy rendering)
# sns.pairplot(train.sample(200), vars=['Temperature','Vibration','Stress index'], hue='Product Quality')

# Section 8: Save/export results and reproducibility

- Save figures using `plt.savefig('figure.png')` in plotting cells.
- Save models with `joblib.dump(model, 'best_model.joblib')`.
- To run this notebook non-interactively and produce outputs, use:

```bash
jupyter nbconvert --to html analysis_and_report.ipynb --execute --ExecutePreprocessor.timeout=600
```

Reproducibility checklist:
- Python version and packages installed from `requirements.txt`.
- Random seed set in the first cell (`np.random.seed(42)`).
- All results were produced using `train_and_tune.py` which saves the final model and `final_submission.csv`.